# TRACERS daily summary plots

衛星とUTC日付区間を指定し、区間内の各日について00–01から23–24まで、1時間ごとの固定10行summary plotを24枚生成する。日付区間は両端を含む。

観測データはUTC日ごとのnative-cadence NetCDF cacheとして一度だけ取得する。CDAWebの元CDF versionを毎回確認し、最新versionのcacheがあれば再downloadせず、古い場合だけ新versionを別名で取得する。SSCWeb軌道はnative 60秒cadenceのまま保存する。各plotは同じdaily cacheを読み、対象1時間へtrimする。データが存在しない行も削除せず、未登録・no data・取得失敗を表示する。

## 設定

`PROBE`は1または2。`START_DATE`と`END_DATE`はUTC日付で、同じ日にすれば1日だけ処理する。取得はhard timeout付きsubprocess、描画はprocessで並列化する。module更新前からkernelを起動している場合は、並列T04実装を読み直すため実行前にkernelをrestartする。

In [ ]:
from pathlib import Path
import os
import sys

WORKSPACE_ROOT = Path.cwd().resolve()
if not (WORKSPACE_ROOT / "module_handmade").exists():
    WORKSPACE_ROOT = WORKSPACE_ROOT.parent
if not (WORKSPACE_ROOT / "module_handmade").exists():
    raise FileNotFoundError("Could not locate module_handmade.")
sys.path.insert(0, str(WORKSPACE_ROOT))

os.environ.setdefault("SPEDAS_DATA_DIR", "/mnt/j/observation_data/")
os.environ.setdefault("SPACEPY", "/tmp/tracers-summary-spacepy")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/tracers-summary-mpl")

PROBE = 2
START_DATE = "2025-10-01"
END_DATE = "2025-12-31"
FORCE_DOWNLOAD = False
SAVE_DAILY_CACHE = True
BIN_INTERVAL_SECONDS = None  # None: native cadence
CONJUGATE_SATELLITES = ["Arase"]  # []なら従来のSSCWeb軌道表示
DOWNLOAD_WORKERS = 4
HTTP_TIMEOUT_SECONDS = 60
DOWNLOAD_ATTEMPT_TIMEOUT_SECONDS = 300
CONJUGACY_ATTEMPT_TIMEOUT_SECONDS = 1800
CONJUGACY_WORKERS = 16
CONJUGACY_CHUNK_SIZE = 16
DOWNLOAD_MAX_ATTEMPTS = 3
DOWNLOAD_RETRY_WAIT_SECONDS = 10
PLOT_WORKERS = 4
SKIP_EXISTING_PLOTS = False

## 日付区間のdaily cache取得と1日24 plot生成

daily cacheは `tracers/ts{probe}/{instrument}/l2/{datatype}/YYYY/MM/*_vX.Y.Z_native.nc`、軌道は `tracers/ts{probe}/orbit/sscweb/YYYY/MM/`、T04共役cacheは `tracers/conjugacy/ts{probe}/{satellite}/t04/YYYY/MM/`、T04のOMNI/Dst入力は `tracers/conjugacy/drivers/t04_omni/YYYY/MM/`、PNGは `tracers/summary_plot/ts{probe}/YYYY/MM/` に保存する。versionなし旧cacheの内部`Data_version`がCDAWeb最新版と一致する場合は再downloadせずversion付き名へ移行する。旧versionは削除しない。CDAWebのversion照会自体が失敗した場合は既存cacheをfreshness未確認として使い、cacheがなければ当該行を取得失敗にする。`CONJUGATE_SATELLITES=["Arase"]`ではAraseをT04+IGRFで瞬時TRACERS高度へmappingし、空リストでは従来表示に戻る。TRACERS自身のSM-MLTとSM磁気緯度はAraseから独立して軌道cacheへ計算するため、Arase取得失敗時もi・j行に残る。T04 traceは`CONJUGACY_CHUNK_SIZE`時刻ずつ、独立したgeopack stateを持つ`CONJUGACY_WORKERS` processで並列実行する。16 worker時は各processのOpenMP/BLAS threadを1へ固定してnested parallelismを避ける。各処理はhard timeoutと指数backoff付きである。`SKIP_EXISTING_PLOTS=True`でもversion確認は行い、cacheより古いPNGだけを再生成する。

In [ ]:
from module_handmade.tracers_summary import generate_summary_plots_for_date_range

figure_paths = generate_summary_plots_for_date_range(
    START_DATE,
    END_DATE,
    probe=PROBE,
    bin_interval_seconds=BIN_INTERVAL_SECONDS,
    conjugate_satellites=CONJUGATE_SATELLITES,
    force_download=FORCE_DOWNLOAD,
    save_daily_cache=SAVE_DAILY_CACHE,
    download_workers=DOWNLOAD_WORKERS,
    http_timeout_seconds=HTTP_TIMEOUT_SECONDS,
    download_attempt_timeout_seconds=DOWNLOAD_ATTEMPT_TIMEOUT_SECONDS,
    conjugacy_attempt_timeout_seconds=CONJUGACY_ATTEMPT_TIMEOUT_SECONDS,
    conjugacy_workers=CONJUGACY_WORKERS,
    conjugacy_chunk_size=CONJUGACY_CHUNK_SIZE,
    download_max_attempts=DOWNLOAD_MAX_ATTEMPTS,
    download_retry_wait_seconds=DOWNLOAD_RETRY_WAIT_SECONDS,
    plot_workers=PLOT_WORKERS,
    skip_existing_plots=SKIP_EXISTING_PLOTS,
    verbose=True,
)
figure_paths

## 固定10行

1. ACE
2. ACI
3. EFI EAC
4. EFI EHF
5. EFI VDC
6. MSC
7. MAGIC
8. Altitude: `(Radius-1)×6371.2 km`
9. MLT: 共役モードではTRACERS実位置とArase T04 footprintのSM-MLT、通常モードではSSCWeb `LT_GM`
10. Signed magnetic latitude: 共役モードでは両者のSM磁気緯度、通常モードではSSCWeb signed `DipInv`

機器データは可能な限りnative cadenceで保存する。共役モードではAraseの北・南footprintを両方cacheし、plot時にTRACERSと同半球のbranchだけを選ぶ。半球切替、MLT wrap、T04入力欠測、trace失敗を線で接続しない。